In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


In [4]:
import json
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # For CUDA error debugging

In [5]:
#Step 1: Load GPT-2 XL
model_name = "gpt2-xl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
tokenizer.pad_token = tokenizer.eos_token

In [7]:
# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1600)
    (wpe): Embedding(1024, 1600)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-47): 48 x GPT2Block(
        (ln_1): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D(nf=4800, nx=1600)
          (c_proj): Conv1D(nf=1600, nx=1600)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=6400, nx=1600)
          (c_proj): Conv1D(nf=1600, nx=6400)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1600, out_features=50257, bias=False)
)

In [8]:
# Step 2: Load and inspect the JSONL file
def load_jsonl(file_path, num_rows=3):
    num_rows = 5
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            data.append(json.loads(line.strip()))
            if i + 1 == num_rows:  # Stop after desired number of rows
                break
    return data

In [9]:
def display_jsonl_structure(file_path):
    data = load_jsonl(file_path)
    if not data:
        print("File is empty or couldn’t be read.")
        return
    
    # Get column names from the first row (assuming consistent structure)
    column_names = list(data[0].keys())
    print("Column Names:", column_names)
    
    # Display first few rows
    print("\nFirst Few Rows:")
    for i, row in enumerate(data):
        print(f"Row {i+1}:")
        for key, value in row.items():
            print(f"  {key}: {value}")
        print()

    return data

In [10]:

# Load and display JSONL
file_path = "/kaggle/input/satact-jsonl/SATACT_v3_trn.jsonl"  # Update this path
print(f"Reading from: {file_path}")

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i < 3:  # Display first 3 lines
                data = json.loads(line.strip())
                print(f"Line {i+1}: {data}")
            else:
                break
except FileNotFoundError:
    print(f"File '{file_path}' not found. Upload your JSONL and check the path.")
except Exception as e:
    print(f"Error: {str(e)}")


#context, question, answerA, answerB, answerC, answerD, correct

Reading from: /kaggle/input/satact-jsonl/SATACT_v3_trn.jsonl
Line 1: {'context': 'While researching a topic, a student has taken the following notes: Ducklings expend up to 62.8% less energy when swimming in a line behind their mother than when swimming alone. The physics behind this energy savings hasn’t always been well understood. Naval architect Zhiming Yuan used computer simulations to study the effect of the mother duck’s wake. The study revealed that ducklings are pushed in a forward direction by the wake’s waves. Yuan determined this push reduces the effect of wave drag on the ducklings by 158%.', 'question': 'The student wants to present the study and its methodology. Which choice most effectively uses relevant information from the notes to accomplish this goal?', 'answerA': 'A study revealed that ducklings, which expend up to 62.8% less energy when swimming in a line behind their mother, also experience 158% less drag.', 'answerB': 'Seeking to understand how ducklings swimmin

The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist 
Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly
covered in boulders, OSIRIS-REx successfully _______ a sample of the surface, gathering pieces of 
it to bring back to Earth. 
Which choice completes the text with the most logical and precise word or phrase? 
A) attached
B) collected
C) followed
D) replaced 

Choice B is the best answer because it most logically completes the text’s discussion of the OSIRIS-REx spacecraft’s contact with the asteroid 101955 Bennu. In this context, “collected” means acquired and took away. The text indicates that although the boulders on the asteroid’s surface caused some unforeseen problems, OSIRIS-REx was able to gather a sample to return to Earth. This context suggests that OSIRIS-REx successfully collected a sample of 101955 Bennu.




Research conducted by planetary scientist Katarina Miljkovic suggests that the Moon’s surface may not accurately _______ early impact events. When the Moon was still forming, its surface was softer, and asteroid or meteoroid impacts would have left less of an impression; thus, evidence of early impacts may no longer be present.

Which choice completes the text with the most logical and precise word or phrase?
A) reflect
B) receive
C) evaluate
D) mimic 

Choice A is the best answer because it most logically completes the text’s
discussion of the Moon’s surface. In this context, “reflect” means show or make
apparent. The text states that because the surface of the Moon was softer when
the Moon was still forming than it is now, early asteroid and meteoroid impacts
“would have left less of an impression” and, as a result, evidence of them may no
longer exist. This context supports the idea that the surface of the Moon may not
accurately show signs of early impact events.

# Stabilizing to produce consistent output by setting random state or seed and producing one answer instead of many
 - by setting do_Sample = False (greedy approach)
 - by limiting max_new_tokens to 100

# One Shot with and without sampling

# Specifics 1: do_sample = False
    - The below function has max_new_tokens = 100 and do_sample = False 
        - (so temperature = 0.5, top_p=0.9 is not needed)
    - eos_token_id=stop_token_ids: added [END] to the example prompt.

In [21]:
#adding [END]
torch.manual_seed(42)
def generate_cot_response_test3(prompt, max_new_tokens=100):  
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
    stop_token_ids = tokenizer.encode("[END]", add_special_tokens=False)
    # Explicitly pass attention_mask to avoid warning
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True, #False is not giving the answer; just produces some text relevant not very meaningful even with increased max_new_tokens
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=stop_token_ids   
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response


# Inline CoT prompt with one-shot example in your style
cot_prompt = (
    "[Example 1]\n"
    "[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist "
    "Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, "
    "OSIRIS-REx successfully _______ a sample of the surface, gathering pieces of it to bring back to Earth.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:"
    "A) attached"
    "B) collected"
    "C) followed"
    "D) replaced"
    "Step-by-step reasoning:\n"
    "Choice B is the best answer because it most logically completes the text’s discussion of the OSIRIS-REx spacecraft’s "
    "contact with the asteroid 101955 Bennu. In this context, “collected” means acquired and took away. The text indicates "
    "that although the boulders on the asteroid’s surface caused some unforeseen problems, OSIRIS-REx was able to gather a "
    "sample to return to Earth. This context suggests that OSIRIS-REx successfully collected a sample of 101955 Bennu.[END]\n\n"
    
    "[Current Problem]\n"
    "[Context]: Research conducted by planetary scientist Katarina Miljkovic suggests that the Moon’s surface may "
    "not accurately _______ early impact events. When the Moon was still forming, its surface was softer, and asteroid or "
    "meteoroid impacts would have left less of an impression; thus, evidence of early impacts may no longer be present.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:\n"
    "A) reflect\n"
    "B) receive\n"
    "C) evaluate\n"
    "D) mimic\n"
    "Step-by-step reasoning:\n"
) 

# Test it
#print(cot_prompt)

# Diagnostic (optional)
inputs = tokenizer(cot_prompt, return_tensors="pt", padding=True)
print(f"Input length: {inputs['input_ids'].shape[1]} tokens")


# Run and display
print("Generating reasoning...\n")
response = generate_cot_response_test3(cot_prompt)
#print("Full Response:\n", response)

#working - print sboth example and current problem with one reasoning: THIS PRINTS THE ENTIRE RESPONSE
print("RESPONSE:\n", response)

Input length: 364 tokens
Generating reasoning...

RESPONSE:
 [Example 1]
[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, OSIRIS-REx successfully _______ a sample of the surface, gathering pieces of it to bring back to Earth.
[Question]: Which choice completes the text with the most logical and precise word or phrase?
[Options]:A) attachedB) collectedC) followedD) replacedStep-by-step reasoning:
Choice B is the best answer because it most logically completes the text’s discussion of the OSIRIS-REx spacecraft’s contact with the asteroid 101955 Bennu. In this context, “collected” means acquired and took away. The text indicates that although the boulders on the asteroid’s surface caused some unforeseen problems, OSIRIS-REx was able to gather a sample to return to Earth. This context suggests that OSIRIS-REx suc

# Evaluating the response

# Specifics 2: do_sample = True
    - The below function has max_new_tokens = 100 and do_sample = False 
        - (so temperature and top_p is needed)
    - eos_token_id=stop_token_ids: added [END] to the example prompt.

In [22]:
#adding [END]
torch.manual_seed(42)
def generate_cot_response_test4(prompt, max_new_tokens=100):  
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
    stop_token_ids = tokenizer.encode("[END]", add_special_tokens=False)
    # Explicitly pass attention_mask to avoid warning
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        temperature=0.1,
        top_p=0.9,
        do_sample=True,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=stop_token_ids   
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response


# Inline CoT prompt with one-shot example in your style
cot_prompt = (
    "[Example 1]\n"
    "[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist "
    "Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, "
    "OSIRIS-REx successfully _______ a sample of the surface, gathering pieces of it to bring back to Earth.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:"
    "A) attached"
    "B) collected"
    "C) followed"
    "D) replaced"
    "Step-by-step reasoning:\n"
    "Choice B is the best answer because it most logically completes the text’s discussion of the OSIRIS-REx spacecraft’s "
    "contact with the asteroid 101955 Bennu. In this context, “collected” means acquired and took away. The text indicates "
    "that although the boulders on the asteroid’s surface caused some unforeseen problems, OSIRIS-REx was able to gather a "
    "sample to return to Earth. This context suggests that OSIRIS-REx successfully collected a sample of 101955 Bennu.[END]\n\n"
    
    "[Current Problem]\n"
    "[Context]: Research conducted by planetary scientist Katarina Miljkovic suggests that the Moon’s surface may "
    "not accurately _______ early impact events. When the Moon was still forming, its surface was softer, and asteroid or "
    "meteoroid impacts would have left less of an impression; thus, evidence of early impacts may no longer be present.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:\n"
    "A) reflect\n"
    "B) receive\n"
    "C) evaluate\n"
    "D) mimic\n"
    "Step-by-step reasoning:\n"
) 

# Test it
#print(cot_prompt)

# Diagnostic (optional)
inputs = tokenizer(cot_prompt, return_tensors="pt", padding=True)
print(f"Input length: {inputs['input_ids'].shape[1]} tokens")


# Run and display
print("Generating reasoning...\n")
response = generate_cot_response_test4(cot_prompt)
#print("Full Response:\n", response)


#working - print sboth example and current problem with one reasoning: THIS PRINTS THE ENTIRE RESPONSE
print("RESPONSE:\n", response)


Input length: 364 tokens
Generating reasoning...

RESPONSE:
 [Example 1]
[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, OSIRIS-REx successfully _______ a sample of the surface, gathering pieces of it to bring back to Earth.
[Question]: Which choice completes the text with the most logical and precise word or phrase?
[Options]:A) attachedB) collectedC) followedD) replacedStep-by-step reasoning:
Choice B is the best answer because it most logically completes the text’s discussion of the OSIRIS-REx spacecraft’s contact with the asteroid 101955 Bennu. In this context, “collected” means acquired and took away. The text indicates that although the boulders on the asteroid’s surface caused some unforeseen problems, OSIRIS-REx was able to gather a sample to return to Earth. This context suggests that OSIRIS-REx suc

# **Two-Shot COT**

# do_sample = False

In [24]:
#adding [END]
torch.manual_seed(42)
def generate_cot_response_test5(prompt, max_new_tokens=100):  
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
    stop_token_ids = tokenizer.encode("[END]", add_special_tokens=False)
    # Explicitly pass attention_mask to avoid warning
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=4,
        no_repeat_ngram_size=7,
        early_stopping=True, #False is not giving the answer; just produces some text relevant not very meaningful even with increased max_new_tokens
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=stop_token_ids   
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Inline CoT prompt with two-shot example in your style
cot_prompt = (
 "[Example 1]\n"
    "[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist "
    "Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, "
    "OSIRIS-REx successfully _______ a sample of the surface, gathering pieces of it to bring back to Earth.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:"
    "A) attached"
    "B) collected"
    "C) followed"
    "D) replaced"
    "Step-by-step reasoning:\n"
    "Choice B is the best answer because it most logically completes the text’s discussion of the OSIRIS-REx spacecraft’s "
    "contact with the asteroid 101955 Bennu. In this context, “collected” means acquired and took away. The text indicates "
    "that although the boulders on the asteroid’s surface caused some unforeseen problems, OSIRIS-REx was able to gather a "
    "sample to return to Earth. This context suggests that OSIRIS-REx successfully collected a sample of 101955 Bennu.[END]\n\n"
 "[Example 2]\n"
    "[Context]: The following text is from the 1913 story “The King’s Coin” by Emily Pauline Johnson, a Kanienkahagen (Mohawk) writer also known as Tekahionwake. Fox-Foot, a young Ojibwe man, is guiding a group of fur traders who are traveling by canoe and suspects that they are being followed. At supper time, Fox-Foot would allow no fire to be built, no landing to be made, no trace of their passing to be left.\n"
    "[Question]: As used in the text, what does the word “trace” most nearly mean?\n"
    "[Options]:"
    "A) Evidence"
    "B) Blemish"
    "C) Amount"
    "D) Sketch\n"
    "Step-by-step reasoning:\n"
    "Choice A is the best answer because the text indicates that Fox-Foot doesn’t let the group build a fire or create a canoe landing when it’s time for supper. This context suggests that he doesn’t want anyone who might be following the group to see any sign of them or their activities. In other words, Fox-Foot doesn’t want there to be any trace, or evidence, of the group’s movements (“their passing”) through the area.[END]\n\n"

 "[Current Problem]\n"
    "[Context]: Research conducted by planetary scientist Katarina Miljkovic suggests that the Moon’s surface may "
    "not accurately _______ early impact events. When the Moon was still forming, its surface was softer, and asteroid or "
    "meteoroid impacts would have left less of an impression; thus, evidence of early impacts may no longer be present.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:"
    "A) reflect"
    "B) receive"
    "C) evaluate"
    "D) mimic\n"
    "Step-by-step reasoning:\n"
) 


# Test it
#print(cot_prompt)


# Diagnostic (optional)
inputs = tokenizer(cot_prompt, return_tensors="pt", padding=True)
print(f"Input length: {inputs['input_ids'].shape[1]} tokens")


# Run and display
print("Generating reasoning...\n")
response = generate_cot_response_test5(cot_prompt)
#print("Full Response:\n", response)
#working - print sboth example and current problem with one reasoning: THIS PRINTS THE ENTIRE RESPONSE
print("RESPONSE:\n", response)


Input length: 632 tokens
Generating reasoning...

RESPONSE:
 [Example 1]
[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, OSIRIS-REx successfully _______ a sample of the surface, gathering pieces of it to bring back to Earth.
[Question]: Which choice completes the text with the most logical and precise word or phrase?
[Options]:A) attachedB) collectedC) followedD) replacedStep-by-step reasoning:
Choice B is the best answer because it most logically completes the text’s discussion of the OSIRIS-REx spacecraft’s contact with the asteroid 101955 Bennu. In this context, “collected” means acquired and took away. The text indicates that although the boulders on the asteroid’s surface caused some unforeseen problems, OSIRIS-REx was able to gather a sample to return to Earth. This context suggests that OSIRIS-REx suc

# Remarks:
- My model QnA has options A and B as correct answers
- That may be the reason for the type of reasoning for above "Current Problem". It starts with saying choices A and B both are best answers and then also says Choice C is better.
- Not very clear reasoning

# Four-shot where options (A,B, C, D) are the right answers
- The "Current Problem" may have A,B,C or D as the right option.
- Choosing questions with A,B,C,D as right answers so that model is not biased/inclined to one of the options or learn any pattern

# Specific 3: do_sample=False with 4-shot

In [28]:
# import torch
# device = torch.device("cpu")

In [17]:
#adding [END]

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch

torch.manual_seed(42)
def generate_cot_response_test6(prompt, max_new_tokens=100):  
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
    stop_token_ids = tokenizer.encode("[END]", add_special_tokens=False)
    # Explicitly pass attention_mask to avoid warning
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        #temperature=0.1,
        #top_p=0.9,
        do_sample=False,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True, #False is not giving the answer; just produces some text relevant not very meaningful even with increased max_new_tokens
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=stop_token_ids   
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Inline CoT prompt with two-shot example in your style
cot_prompt = (
 "[Example 1]\n"
    "[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist "
    "Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, "
    "OSIRIS-REx successfully _ a sample of the surface, gathering pieces of it to bring back to Earth.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:"
    "A) attached"
    "B) collected"
    "C) followed"
    "D) replaced"
    "Step-by-step reasoning:\n"
    "Choice B is the best answer because it most logically completes the text’s discussion of the OSIRIS-REx spacecraft’s "
    "contact with the asteroid 101955 Bennu. In this context, “collected” means acquired and took away. The text indicates "
    "that although the boulders on the asteroid’s surface caused some unforeseen problems, OSIRIS-REx was able to gather a "
    "sample to return to Earth. This context suggests that OSIRIS-REx successfully collected a sample of 101955 Bennu.[END]\n\n"
 # "[Example 2]\n"
 #    "[Context]: The following text is from the 1913 story “The King’s Coin” by Emily Pauline Johnson, a Kanienkahagen (Mohawk) writer also known as Tekahionwake. Fox-Foot, a young Ojibwe man, is guiding a group of fur traders who are traveling by canoe and suspects that they are being followed. At supper time, Fox-Foot would allow no fire to be built, no landing to be made, no trace of their passing to be left.\n"
 #    "[Question]: As used in the text, what does the word “trace” most nearly mean?\n"
 #    "[Options]:"
 #    "A) Evidence"
 #    "B) Blemish"
 #    "C) Amount"
 #    "D) Sketch\n"
 #    "Step-by-step reasoning:\n"
 #    "Choice A is the best answer because the text indicates that Fox-Foot doesn’t let the group build a fire or create a canoe landing when it’s time for supper. This context suggests that he doesn’t want anyone who might be following the group to see any sign of them or their activities. In other words, Fox-Foot doesn’t want there to be any trace, or evidence, of the group’s movements (“their passing”) through the area.[END]\n\n"

 "[Example 3]\n"
    "[Context]: The narrator of Charlotte Perkins Gilman’s 1892 short story “The Yellow Wallpaper” expresses mixed feelings about her surroundings._\n"
    "[Question]: Which quotation from the text most effectively illustrates the claim about the narrator’s feelings?\n"
    "[Options]:"
    "A) This wallpaper has a kind of sub-pattern in a different shade, a particularly irritating one, for you can only see it in certain lights, and not clearly then."
    "B) By moonlight—the moon shines in all night when there is a moon—I wouldn’t know it was the same paper."
    "C) I’m really getting quite fond of the big room, all but that horrid [wall]paper."
    "D) The color is repellant, almost revolting; a smouldering, unclean yellow, strangely faded by the slow-turning sunlight.”\n"
    "Step-by-step reasoning:\n"
    "Choice C is the best answer because it most effectively illustrates the claim that the narrator of “The Yellow Wallpaper” has mixed feelings about her surroundings. She says she is “really getting quite fond of the big room,” a positive sentiment, but also describes the room’s wallpaper as “horrid,” a negative sentiment. Since some of her feelings about her surroundings are positive and others are negative, they are best described as mixed.[END]\n\n"

 # "[Example 4]\n"
 #    "[Context]: Engineer Aroba Saleem and colleagues demonstrated that the integrity of underground metal pipes can be assessed without unearthing the pipes by using a particular property of some metals.\n"
 #    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
 #    "[Options]:\n"
 #    "A) hypothesized"
 #    "B) discounted"
 #    "C) redefined"
 #    "D) exploited\n"
 #    "Step-by-step reasoning:\n"
 #    "Choice D is the best answer because it most logically completes the text’s discussion of using magnetism to detect stress in buried metal pipes. In this context, “exploited” means made productive use of. The text indicates that the magnetic fields of some metals change under stress and that Saleem and colleagues showed that it is possible to measure those changes from a distance, thereby demonstrating that the integrity of underground metal pipes can be evaluated without having to unearth them.[END]\n\n"
    
 "[Current Problem]\n"
    "[Context]: Research conducted by planetary scientist Katarina Miljkovic suggests that the Moon’s surface may "
    "not accurately _ early impact events. When the Moon was still forming, its surface was softer, and asteroid or "
    "meteoroid impacts would have left less of an impression; thus, evidence of early impacts may no longer be present.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:"
    "A) reflect"
    "B) receive"
    "C) evaluate"
    "D) mimic\n"
    "Step-by-step reasoning:\n"
) 


# Test it
#print(cot_prompt)


# Diagnostic (optional)
inputs = tokenizer(cot_prompt, return_tensors="pt", padding=True)
print(f"Input length: {inputs['input_ids'].shape[1]} tokens")


# Run and display
print("Generating reasoning...\n")
response = generate_cot_response_test6(cot_prompt)
#print("Full Response:\n", response)
#working - print sboth example and current problem with one reasoning: THIS PRINTS THE ENTIRE RESPONSE
print("RESPONSE:\n", response)

Input length: 655 tokens
Generating reasoning...

RESPONSE:
 [Example 1]
[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, OSIRIS-REx successfully _ a sample of the surface, gathering pieces of it to bring back to Earth.
[Question]: Which choice completes the text with the most logical and precise word or phrase?
[Options]:A) attachedB) collectedC) followedD) replacedStep-by-step reasoning:
Choice B is the best answer because it most logically completes the text’s discussion of the OSIRIS-REx spacecraft’s contact with the asteroid 101955 Bennu. In this context, “collected” means acquired and took away. The text indicates that although the boulders on the asteroid’s surface caused some unforeseen problems, OSIRIS-REx was able to gather a sample to return to Earth. This context suggests that OSIRIS-REx successfu

# Remarks:
- 4-shot exmaple unable to run as it exceeds 1024 limit.
- It can work only for very small contexts
- Above I have removed contexts to lomit within 1024

# Specific 4: do_sample = True

In [15]:
#adding [END]

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch

torch.manual_seed(42)
def generate_cot_response_test6(prompt, max_new_tokens=100):  
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
    stop_token_ids = tokenizer.encode("[END]", add_special_tokens=False)
    # Explicitly pass attention_mask to avoid warning
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        temperature=0.05,
        top_p=0.9,
        do_sample=True,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True, #False is not giving the answer; just produces some text relevant not very meaningful even with increased max_new_tokens
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=stop_token_ids   
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Inline CoT prompt with two-shot example in your style
cot_prompt = (
 "[Example 1]\n"
    "[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist "
    "Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, "
    "OSIRIS-REx successfully _ a sample of the surface, gathering pieces of it to bring back to Earth.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:"
    "A) attached"
    "B) collected"
    "C) followed"
    "D) replaced"
    "Step-by-step reasoning:\n"
    "Choice B is the best answer because it most logically completes the text’s discussion of the OSIRIS-REx spacecraft’s "
    "contact with the asteroid 101955 Bennu. In this context, “collected” means acquired and took away. The text indicates "
    "that although the boulders on the asteroid’s surface caused some unforeseen problems, OSIRIS-REx was able to gather a "
    "sample to return to Earth. This context suggests that OSIRIS-REx successfully collected a sample of 101955 Bennu.[END]\n\n"
 # "[Example 2]\n"
 #    "[Context]: The following text is from the 1913 story “The King’s Coin” by Emily Pauline Johnson, a Kanienkahagen (Mohawk) writer also known as Tekahionwake. Fox-Foot, a young Ojibwe man, is guiding a group of fur traders who are traveling by canoe and suspects that they are being followed. At supper time, Fox-Foot would allow no fire to be built, no landing to be made, no trace of their passing to be left.\n"
 #    "[Question]: As used in the text, what does the word “trace” most nearly mean?\n"
 #    "[Options]:"
 #    "A) Evidence"
 #    "B) Blemish"
 #    "C) Amount"
 #    "D) Sketch\n"
 #    "Step-by-step reasoning:\n"
 #    "Choice A is the best answer because the text indicates that Fox-Foot doesn’t let the group build a fire or create a canoe landing when it’s time for supper. This context suggests that he doesn’t want anyone who might be following the group to see any sign of them or their activities. In other words, Fox-Foot doesn’t want there to be any trace, or evidence, of the group’s movements (“their passing”) through the area.[END]\n\n"

 "[Example 3]\n"
    "[Context]: The narrator of Charlotte Perkins Gilman’s 1892 short story “The Yellow Wallpaper” expresses mixed feelings about her surroundings._\n"
    "[Question]: Which quotation from the text most effectively illustrates the claim about the narrator’s feelings?\n"
    "[Options]:"
    "A) This wallpaper has a kind of sub-pattern in a different shade, a particularly irritating one, for you can only see it in certain lights, and not clearly then."
    "B) By moonlight—the moon shines in all night when there is a moon—I wouldn’t know it was the same paper."
    "C) I’m really getting quite fond of the big room, all but that horrid [wall]paper."
    "D) The color is repellant, almost revolting; a smouldering, unclean yellow, strangely faded by the slow-turning sunlight.”\n"
    "Step-by-step reasoning:\n"
    "Choice C is the best answer because it most effectively illustrates the claim that the narrator of “The Yellow Wallpaper” has mixed feelings about her surroundings. She says she is “really getting quite fond of the big room,” a positive sentiment, but also describes the room’s wallpaper as “horrid,” a negative sentiment. Since some of her feelings about her surroundings are positive and others are negative, they are best described as mixed.[END]\n\n"

 # "[Example 4]\n"
 #    "[Context]: Engineer Aroba Saleem and colleagues demonstrated that the integrity of underground metal pipes can be assessed without unearthing the pipes by using a particular property of some metals.\n"
 #    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
 #    "[Options]:\n"
 #    "A) hypothesized"
 #    "B) discounted"
 #    "C) redefined"
 #    "D) exploited\n"
 #    "Step-by-step reasoning:\n"
 #    "Choice D is the best answer because it most logically completes the text’s discussion of using magnetism to detect stress in buried metal pipes. In this context, “exploited” means made productive use of. The text indicates that the magnetic fields of some metals change under stress and that Saleem and colleagues showed that it is possible to measure those changes from a distance, thereby demonstrating that the integrity of underground metal pipes can be evaluated without having to unearth them.[END]\n\n"
    
 "[Current Problem]\n"
    "[Context]: Research conducted by planetary scientist Katarina Miljkovic suggests that the Moon’s surface may "
    "not accurately _ early impact events. When the Moon was still forming, its surface was softer, and asteroid or "
    "meteoroid impacts would have left less of an impression; thus, evidence of early impacts may no longer be present.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:"
    "A) reflect"
    "B) receive"
    "C) evaluate"
    "D) mimic\n"
    "Step-by-step reasoning:\n"
) 


# Test it
#print(cot_prompt)


# Diagnostic (optional)
inputs = tokenizer(cot_prompt, return_tensors="pt", padding=True)
print(f"Input length: {inputs['input_ids'].shape[1]} tokens")


# Run and display
print("Generating reasoning...\n")
response = generate_cot_response_test6(cot_prompt)
#print("Full Response:\n", response)

#working - print sboth example and current problem with one reasoning: THIS PRINTS THE ENTIRE RESPONSE
print("RESPONSE:\n", response)

Input length: 655 tokens
Generating reasoning...

RESPONSE:
 [Example 1]
[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, OSIRIS-REx successfully _ a sample of the surface, gathering pieces of it to bring back to Earth.
[Question]: Which choice completes the text with the most logical and precise word or phrase?
[Options]:A) attachedB) collectedC) followedD) replacedStep-by-step reasoning:
Choice B is the best answer because it most logically completes the text’s discussion of the OSIRIS-REx spacecraft’s contact with the asteroid 101955 Bennu. In this context, “collected” means acquired and took away. The text indicates that although the boulders on the asteroid’s surface caused some unforeseen problems, OSIRIS-REx was able to gather a sample to return to Earth. This context suggests that OSIRIS-REx successfu

# Remarks:
- three or four shot doesn't work because of the 1024 token limit.
- Two - shot is not a great way here as you can see. There is no clear explanation
- Can try using summaries of these context

# Using compressed prompts from LLMLingua to try four-shot COT

# Specifics 5: Prompt compressed and do_sample = False
- 4-shot didn't run exceeded 1024 limit. Tried 3-shot

In [10]:
#adding [END]

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch

torch.manual_seed(42)
def generate_cot_response_test6(prompt, max_new_tokens=100):  
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
    stop_token_ids = tokenizer.encode("[END]", add_special_tokens=False)
    # Explicitly pass attention_mask to avoid warning
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True, #False is not giving the answer; just produces some text relevant not very meaningful even with increased max_new_tokens
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=stop_token_ids   
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response


cot_prompt = (
    "[Example 1]\n"
    "[Context]: Coleridge-Taylor classical composer England toured US three times early 1900s child West African English mother emphasized mixed-race ancestry referred Anglo-African[BLANK]incorporated African music classical compositions.\n"
    "[Question]: Which choice completes the text with the most logical transition?\n"
    "[Options]:\n"
    "A) In addition,\n"
    "B) Actually,\n"
    "C) However,\n"
    "D) Regardless,\n"
    "Step-by-step reasoning:\n"
    "Choice A is the best answer. \"In addition\" logically signals that the detail in this sentence—that Coleridge-Taylor included traditional African music in his classical compositions—adds to the information in the previous sentence. Specifically, the previous sentence indicates one way in which Coleridge-Taylor emphasized his mixed-race ancestry, and the claim that follows indicates a second, additional way.[END]\n\n"
    
    "[Example 2]\n"
    "[Context]: thousands humans used domesticated goats hircus clear land unwanted vegetation diets goats[BLANK]devour shrubs weeds leaving no part plant unconsumed.\n"
    "[Question]: Which choice completes the text so that it conforms to the conventions of Standard English?\n"
    "[Options]:\n"
    "A) indiscriminate and\n"
    "B) indiscriminate,\n"
    "C) indiscriminate\n"
    "D) indiscriminate:\n"
    "Step-by-step reasoning:\n"
    "Choice D is the best answer. The convention being tested is punctuation use between two main clauses. In this choice, a colon is correctly used to mark the boundary between one main clause (“goats are notoriously indiscriminate”) and another main clause (“they will devour all kinds of shrubs and weeds”) and to introduce the following explanation of goats’ nondiscriminatory behavior when it comes to what they eat.[END]\n\n"
    
    "[Example 3]\n"
    "[Context]: clichéd suburban power lines John Ashbery’s 2004 poem “Ignorance Law No barren analysis[BLANK]critic Lauren Berlant finds ground first two stanzas devoting chapter deciphering default space” Ashbery\n"
    "[Question]: Which choice completes the text with the most logical transition?\n"
    "[Options]:\n"
    "A) Likewise,\n"
    "B) Nonetheless,\n"
    "C) In turn,\n"
    "D) That is,\n"
    "Step-by-step reasoning:\n"
    "Choice B is the best answer. \"Nonetheless\" is a transition that indicates disagreement. The first sentence describes the unlikelihood of finding much for critical analysis in Ashbery's poem (\"barren terrain\"), while the second sentence describes how Berlant did in fact find much to analyze in Ashbery's poem (\"fertile ground\"), so the transition \"nonetheless\" fits perfectly.[END]\n\n"
    
    # "[Example 4]\n"
    # "[Context]: Yellow 1892 short story Charlotte Perkins narrator expresses mixed feelings surroundings[BLANK].\n"
    # "[Question]: Which quotation from “The Yellow Wallpaper” most effectively illustrates the claim?\n"
    # "[Options]:\n"
    # "A) This wallpaper has a kind of sub-pattern in a different shade, a particularly irritating one, for you can only see it in certain lights, and not clearly then.\n"
    # "B) By moonlight—the moon shines in all night when there is a moon—I wouldn’t know it was the same paper.\n"
    # "C) I’m really getting quite fond of the big room, all but that horrid [wall]paper.\n"
    # "D) The color is repellant, almost revolting; a smouldering, unclean yellow, strangely faded by the slow-turning sunlight.\n"
    # "Step-by-step reasoning:\n"
    # "Choice C is the best answer because it most effectively illustrates the claim that the narrator of “The Yellow Wallpaper” has mixed feelings about her surroundings. She says she is “really getting quite fond of the big room,” a positive sentiment, but also describes the room’s wallpaper as “horrid,” a negative sentiment. Since some of her feelings about her surroundings are positive and others are negative, they are best described as mixed.[END]\n\n"
    
    "[Current Problem]\n"
    "[Context]: Research conducted by planetary scientist Katarina Miljkovic suggests that the Moon’s surface may "
    "not accurately _ early impact events. When the Moon was still forming, its surface was softer, and asteroid or "
    "meteoroid impacts would have left less of an impression; thus, evidence of early impacts may no longer be present.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:\n"
    "A) reflect\n"
    "B) receive\n"
    "C) evaluate\n"
    "D) mimic\n"
    "Step-by-step reasoning:\n"
)

# Test it
#print(cot_prompt)


# Diagnostic (optional)
inputs = tokenizer(cot_prompt, return_tensors="pt", padding=True)
print(f"Input length: {inputs['input_ids'].shape[1]} tokens")


# Run and display
print("Generating reasoning...\n")
response = generate_cot_response_test6(cot_prompt)
#print("Full Response:\n", response)

#working - print sboth example and current problem with one reasoning: THIS PRINTS THE ENTIRE RESPONSE
print("RESPONSE:\n", response)

Input length: 684 tokens
Generating reasoning...

RESPONSE:
 [Example 1]
[Context]: Coleridge-Taylor classical composer England toured US three times early 1900s child West African English mother emphasized mixed-race ancestry referred Anglo-African[BLANK]incorporated African music classical compositions.
[Question]: Which choice completes the text with the most logical transition?
[Options]:
A) In addition,
B) Actually,
C) However,
D) Regardless,
Step-by-step reasoning:
Choice A is the best answer. "In addition" logically signals that the detail in this sentence—that Coleridge-Taylor included traditional African music in his classical compositions—adds to the information in the previous sentence. Specifically, the previous sentence indicates one way in which Coleridge-Taylor emphasized his mixed-race ancestry, and the claim that follows indicates a second, additional way.[END]

[Example 2]
[Context]: thousands humans used domesticated goats hircus clear land unwanted vegetation diets 

# Remarks
- 4 shot exceeded 1024 limit, 3 shot executed
- 3-shot examples used compressed prompt, "Current Problem" used original prompt


# Compress the prompt for the "Current Problem"

In [13]:
#adding [END]

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch

torch.manual_seed(42)
def generate_cot_response_test6(prompt, max_new_tokens=150):  
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
    stop_token_ids = tokenizer.encode("[END]", add_special_tokens=False)
    # Explicitly pass attention_mask to avoid warning
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True, #False is not giving the answer; just produces some text relevant not very meaningful even with increased max_new_tokens
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=stop_token_ids   
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response


cot_prompt = (
    "[Example 1]\n"
    "[Context]: Coleridge-Taylor classical composer England toured US three times early 1900s child West African English mother emphasized mixed-race ancestry referred Anglo-African[BLANK]incorporated African music classical compositions.\n"
    "[Question]: Which choice completes the text with the most logical transition?\n"
    "[Options]:\n"
    "A) In addition,\n"
    "B) Actually,\n"
    "C) However,\n"
    "D) Regardless,\n"
    "Step-by-step reasoning:\n"
    "Choice A is the best answer. \"In addition\" logically signals that the detail in this sentence—that Coleridge-Taylor included traditional African music in his classical compositions—adds to the information in the previous sentence. Specifically, the previous sentence indicates one way in which Coleridge-Taylor emphasized his mixed-race ancestry, and the claim that follows indicates a second, additional way.[END]\n\n"
    
    "[Example 2]\n"
    "[Context]: thousands humans used domesticated goats hircus clear land unwanted vegetation diets goats[BLANK]devour shrubs weeds leaving no part plant unconsumed.\n"
    "[Question]: Which choice completes the text so that it conforms to the conventions of Standard English?\n"
    "[Options]:\n"
    "A) indiscriminate and\n"
    "B) indiscriminate,\n"
    "C) indiscriminate\n"
    "D) indiscriminate:\n"
    "Step-by-step reasoning:\n"
    "Choice D is the best answer. The convention being tested is punctuation use between two main clauses. In this choice, a colon is correctly used to mark the boundary between one main clause (“goats are notoriously indiscriminate”) and another main clause (“they will devour all kinds of shrubs and weeds”) and to introduce the following explanation of goats’ nondiscriminatory behavior when it comes to what they eat.[END]\n\n"
    
    "[Example 3]\n"
    "[Context]: clichéd suburban power lines John Ashbery’s 2004 poem “Ignorance Law No barren analysis[BLANK]critic Lauren Berlant finds ground first two stanzas devoting chapter deciphering default space” Ashbery\n"
    "[Question]: Which choice completes the text with the most logical transition?\n"
    "[Options]:\n"
    "A) Likewise,\n"
    "B) Nonetheless,\n"
    "C) In turn,\n"
    "D) That is,\n"
    "Step-by-step reasoning:\n"
    "Choice B is the best answer. \"Nonetheless\" is a transition that indicates disagreement. The first sentence describes the unlikelihood of finding much for critical analysis in Ashbery's poem (\"barren terrain\"), while the second sentence describes how Berlant did in fact find much to analyze in Ashbery's poem (\"fertile ground\"), so the transition \"nonetheless\" fits perfectly.[END]\n\n"
    
    # "[Example 4]\n"
    # "[Context]: Yellow 1892 short story Charlotte Perkins narrator expresses mixed feelings surroundings[BLANK].\n"
    # "[Question]: Which quotation from “The Yellow Wallpaper” most effectively illustrates the claim?\n"
    # "[Options]:\n"
    # "A) This wallpaper has a kind of sub-pattern in a different shade, a particularly irritating one, for you can only see it in certain lights, and not clearly then.\n"
    # "B) By moonlight—the moon shines in all night when there is a moon—I wouldn’t know it was the same paper.\n"
    # "C) I’m really getting quite fond of the big room, all but that horrid [wall]paper.\n"
    # "D) The color is repellant, almost revolting; a smouldering, unclean yellow, strangely faded by the slow-turning sunlight.\n"
    # "Step-by-step reasoning:\n"
    # "Choice C is the best answer because it most effectively illustrates the claim that the narrator of “The Yellow Wallpaper” has mixed feelings about her surroundings. She says she is “really getting quite fond of the big room,” a positive sentiment, but also describes the room’s wallpaper as “horrid,” a negative sentiment. Since some of her feelings about her surroundings are positive and others are negative, they are best described as mixed.[END]\n\n"

    "[Current Problem]\n"
    "[Context]: 2018 researchers led Dr. Caitlin Whalen compiled ocean mixing rates past two decades novel data set current-driven mixing _ impact on distribution heat nutrients in ocean.\n"
    "[Question]: Which choice completes the text so that it conforms to the conventions of Standard English? \n"
    "[Options]:\n"
    "A) regions,\n"
    "B) regions:\n"
    "C) regions;\n"
    "D) regions\n"
    "Step-by-step reasoning:\n"
)

# Test it
#print(cot_prompt)


# Diagnostic (optional)
inputs = tokenizer(cot_prompt, return_tensors="pt", padding=True)
print(f"Input length: {inputs['input_ids'].shape[1]} tokens")


# Run and display
print("Generating reasoning...\n")
response = generate_cot_response_test6(cot_prompt)
#print("Full Response:\n", response)
#working - print sboth example and current problem with one reasoning: THIS PRINTS THE ENTIRE RESPONSE
print("RESPONSE:\n", response)

Input length: 657 tokens
Generating reasoning...

RESPONSE:
 [Example 1]
[Context]: Coleridge-Taylor classical composer England toured US three times early 1900s child West African English mother emphasized mixed-race ancestry referred Anglo-African[BLANK]incorporated African music classical compositions.
[Question]: Which choice completes the text with the most logical transition?
[Options]:
A) In addition,
B) Actually,
C) However,
D) Regardless,
Step-by-step reasoning:
Choice A is the best answer. "In addition" logically signals that the detail in this sentence—that Coleridge-Taylor included traditional African music in his classical compositions—adds to the information in the previous sentence. Specifically, the previous sentence indicates one way in which Coleridge-Taylor emphasized his mixed-race ancestry, and the claim that follows indicates a second, additional way.[END]

[Example 2]
[Context]: thousands humans used domesticated goats hircus clear land unwanted vegetation diets 

# Remarks:
- Even with compressed contexts for all examples and including the "Current Problem", its exceeding 1024 limit.
- So here I have removed one of the example and able to produce some reasoning. The reasoning produced is still not up to the mark. Either it says two options are better or 3 options are better, not definite about it.

# Use one example, each time give one of the options as the correct answer.
# Three-shot
- do_sample = False

In [14]:
#adding [END]

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch

torch.manual_seed(42)
def generate_cot_response_test6(prompt, max_new_tokens=100):  
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
    stop_token_ids = tokenizer.encode("[END]", add_special_tokens=False)
    # Explicitly pass attention_mask to avoid warning
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True, #False is not giving the answer; just produces some text relevant not very meaningful even with increased max_new_tokens
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=stop_token_ids   
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Inline CoT prompt with two-shot example in your style
cot_prompt = (
 "[Example 1]\n"
    "[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist "
    "Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, "
    "OSIRIS-REx successfully _ a sample of the surface, gathering pieces of it to bring back to Earth.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:"
    "A) attached"
    "B) collected"
    "C) followed"
    "D) replaced"
    "Step-by-step reasoning:\n"
    "Choice B is the best answer because it most logically completes the text’s discussion of the OSIRIS-REx spacecraft’s "
    "contact with the asteroid 101955 Bennu. In this context, “collected” means acquired and took away. The text indicates "
    "that although the boulders on the asteroid’s surface caused some unforeseen problems, OSIRIS-REx was able to gather a "
    "sample to return to Earth. This context suggests that OSIRIS-REx successfully collected a sample of 101955 Bennu.[END]\n\n"
  
"[Example 2]\n"
    "[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist "
    "Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, "
    "OSIRIS-REx successfully _ a sample of the surface, gathering pieces of it to bring back to Earth.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:"
    "A) attached"
    "B) collected"
    "C) followed"
    "D) replaced"
    "Step-by-step reasoning:\n"
    "Choice A is the best answer because it most logically completes the text’s discussion.[END]\n\n"
    
 "[Example 3]\n"
    "[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist "
    "Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, "
    "OSIRIS-REx successfully _ a sample of the surface, gathering pieces of it to bring back to Earth.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:"
    "A) attached"
    "B) collected"
    "C) followed"
    "D) replaced"
    "Step-by-step reasoning:\n"
    "Choice D is the best answer because it most logically completes the text’s discussion.[END]\n\n"

 # "[Example 1]\n"
 #    "[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist "
 #    "Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, "
 #    "OSIRIS-REx successfully _ a sample of the surface, gathering pieces of it to bring back to Earth.\n"
 #    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
 #    "[Options]:"
 #    "A) attached"
 #    "B) collected"
 #    "C) followed"
 #    "D) replaced"
 #    "Step-by-step reasoning:\n"
 #    "Choice C is the best answer because it most logically completes the text’s discussion.[END]\n\n"
    
 "[Current Problem]\n"
    "[Context]: Research conducted by planetary scientist Katarina Miljkovic suggests that the Moon’s surface may "
    "not accurately _ early impact events. When the Moon was still forming, its surface was softer, and asteroid or "
    "meteoroid impacts would have left less of an impression; thus, evidence of early impacts may no longer be present.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:"
    "A) reflect"
    "B) receive"
    "C) evaluate"
    "D) mimic\n"
    "Step-by-step reasoning:\n"
) 


# Test it
#print(cot_prompt)


# Diagnostic (optional)
inputs = tokenizer(cot_prompt, return_tensors="pt", padding=True)
print(f"Input length: {inputs['input_ids'].shape[1]} tokens")


# Run and display
print("Generating reasoning...\n")
response = generate_cot_response_test6(cot_prompt)
#print("Full Response:\n", response)

#working - print sboth example and current problem with one reasoning: THIS PRINTS THE ENTIRE RESPONSE
print("RESPONSE:\n", response)

Input length: 648 tokens
Generating reasoning...

RESPONSE:
 [Example 1]
[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, OSIRIS-REx successfully _ a sample of the surface, gathering pieces of it to bring back to Earth.
[Question]: Which choice completes the text with the most logical and precise word or phrase?
[Options]:A) attachedB) collectedC) followedD) replacedStep-by-step reasoning:
Choice B is the best answer because it most logically completes the text’s discussion of the OSIRIS-REx spacecraft’s contact with the asteroid 101955 Bennu. In this context, “collected” means acquired and took away. The text indicates that although the boulders on the asteroid’s surface caused some unforeseen problems, OSIRIS-REx was able to gather a sample to return to Earth. This context suggests that OSIRIS-REx successfu

# Remark:
- One example, repeated 3 times with different options as the right answer.
- The COT for the "Current Problem" is not too clear
- Limit exceeds(>1024) - was able to do one example, with 3 different options instead of 4
- 

# Three-shot
- do_sample = True


In [18]:
#adding [END]

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch

torch.manual_seed(42)
def generate_cot_response_test6(prompt, max_new_tokens=250):  
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
    stop_token_ids = tokenizer.encode("[END]", add_special_tokens=False)
    # Explicitly pass attention_mask to avoid warning
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        temperature=0.05,
        top_p=0.9,
        do_sample=True,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True, #False is not giving the answer; just produces some text relevant not very meaningful even with increased max_new_tokens
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=stop_token_ids   
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Inline CoT prompt with two-shot example in your style
cot_prompt = (
 "[Example 1]\n"
    "[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist "
    "Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, "
    "OSIRIS-REx successfully _ a sample of the surface, gathering pieces of it to bring back to Earth.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:"
    "A) attached"
    "B) collected"
    "C) followed"
    "D) replaced"
    "Step-by-step reasoning:\n"
    "Choice B is the best answer because it most logically completes the text’s discussion of the OSIRIS-REx spacecraft’s "
    "contact with the asteroid 101955 Bennu. In this context, “collected” means acquired and took away. The text indicates "
    "that although the boulders on the asteroid’s surface caused some unforeseen problems, OSIRIS-REx was able to gather a "
    "sample to return to Earth. This context suggests that OSIRIS-REx successfully collected a sample of 101955 Bennu.[END]\n\n"
  
"[Example 2]\n"
    "[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist "
    "Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, "
    "OSIRIS-REx successfully _ a sample of the surface, gathering pieces of it to bring back to Earth.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:"
    "A) attached"
    "B) collected"
    "C) followed"
    "D) replaced"
    "Step-by-step reasoning:\n"
    "Choice A is the best answer because it most logically completes the text’s discussion.[END]\n\n"
    
 "[Example 3]\n"
    "[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist "
    "Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, "
    "OSIRIS-REx successfully _ a sample of the surface, gathering pieces of it to bring back to Earth.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:"
    "A) attached"
    "B) collected"
    "C) followed"
    "D) replaced"
    "Step-by-step reasoning:\n"
    "Choice D is the best answer because it most logically completes the text’s discussion.[END]\n\n"

 "[Current Problem]\n"
    "[Context]: Research conducted by planetary scientist Katarina Miljkovic suggests that the Moon’s surface may "
    "not accurately _ early impact events. When the Moon was still forming, its surface was softer, and asteroid or "
    "meteoroid impacts would have left less of an impression; thus, evidence of early impacts may no longer be present.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:"
    "A) reflect"
    "B) receive"
    "C) evaluate"
    "D) mimic\n"
    "Step-by-step reasoning:\n"
) 


# Test it
#print(cot_prompt)


# Diagnostic (optional)
inputs = tokenizer(cot_prompt, return_tensors="pt", padding=True)
print(f"Input length: {inputs['input_ids'].shape[1]} tokens")


# Run and display
print("Generating reasoning...\n")
response = generate_cot_response_test6(cot_prompt)
#print("Full Response:\n", response)

#working - print sboth example and current problem with one reasoning: THIS PRINTS THE ENTIRE RESPONSE
print("RESPONSE:\n", response)

Input length: 648 tokens
Generating reasoning...

RESPONSE:
 [Example 1]
[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, OSIRIS-REx successfully _ a sample of the surface, gathering pieces of it to bring back to Earth.
[Question]: Which choice completes the text with the most logical and precise word or phrase?
[Options]:A) attachedB) collectedC) followedD) replacedStep-by-step reasoning:
Choice B is the best answer because it most logically completes the text’s discussion of the OSIRIS-REx spacecraft’s contact with the asteroid 101955 Bennu. In this context, “collected” means acquired and took away. The text indicates that although the boulders on the asteroid’s surface caused some unforeseen problems, OSIRIS-REx was able to gather a sample to return to Earth. This context suggests that OSIRIS-REx successfu

# Remarks:
- Was able to run 3-shot: It depends on the length of the context
- This was using do_sample = True: So the response has introdiced new symbols "||" and is not well structured.
- No definite correct answer at the end of COT run